In [4]:
file_path = "api.py"

with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

lines = content.splitlines()
num_lines = len(lines)

print(f"Total lines: {num_lines}")
# print(f"First 7 lines: \n{line: for line in lines[:8]}")
print(lines[:7])

Total lines: 180
['"""', 'requests.api', '~~~~~~~~~~~~', '', 'This module implements the Requests API.', '', ':copyright: (c) 2012 by Kenneth Reitz.']


In [9]:
from pathlib import Path
from typing import Iterable, TypedDict, Any


class Chunk(TypedDict):
    chunk_id: Any
    source_file: str
    start_line: int
    end_line: int
    text: str
# Iterable[Path | str]

def chunk_files(files_path: Path, chunk_size: int = 40, overlap: int = 5,) -> list[Chunk]:
    """
    Split each file into overlapping line-based chunks.

    Sliding window behavior:
    - chunk size = `chunk_size`
    - overlap = `overlap`
    - stride = `chunk_size - overlap`
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if overlap < 0:
        raise ValueError("overlap must be >= 0")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    stride = chunk_size - overlap
    chunks: list[Chunk] = []
    chunk_id = 0

    path = Path(files_path)
    file_name = str(path)

    # Stage 1: read file and split into lines
    with path.open("r", encoding="utf-8") as f:
        content = f.read()
    lines = content.splitlines()
    n = len(lines)

    # Restart window index for each new file
    i = 0
    while i < n:
        # Stage 4: take one window
        chunk_lines = lines[i : i + chunk_size]

        # Stage 5: build metadata + text payload
        chunk: Chunk = {
            "chunk_id": f"{file_name}_{chunk_id}",
            "source_file": str(path),
            "start_line": i,
            "end_line": min(i + chunk_size, n) - 1,
            "text": "\n".join(chunk_lines),
            }
        chunks.append(chunk)

        if chunk["end_line"] == n - 1:
            break

        # Stage 6: move by stride, not chunk_size
        i += stride
        chunk_id += 1

    return chunks

In [12]:
file = Path("api.py")
chunked = chunk_files(file)
print(f"Total chunks in api.py: {len(chunked)}")
for chunk in chunked[:10]:
    print(chunk)
    print()

Total chunks in api.py: 5
{'chunk_id': 'api.py_0', 'source_file': 'api.py', 'start_line': 0, 'end_line': 39, 'text': '"""\nrequests.api\n~~~~~~~~~~~~\n\nThis module implements the Requests API.\n\n:copyright: (c) 2012 by Kenneth Reitz.\n:license: Apache2, see LICENSE for more details.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import TYPE_CHECKING\n\nfrom . import sessions\nfrom .models import Response\n\nif TYPE_CHECKING:\n    from typing_extensions import Unpack\n\n    from . import _types as _t\n\n\ndef request(\n    method: str, url: _t.UriType, **kwargs: Unpack[_t.RequestKwargs]\n) -> Response:\n    """Constructs and sends a :class:`Request <Request>`.\n\n    :param method: method for the new :class:`Request` object: ``GET``, ``OPTIONS``, ``HEAD``, ``POST``, ``PUT``, ``PATCH``, or ``DELETE``.\n    :param url: URL for the new :class:`Request` object.\n    :param params: (optional) Dictionary, list of tuples or bytes to send\n        in the query string for the :class

In [13]:
file = Path("exceptions.py")
exceptions_chunked = chunk_files(file)

print(f"Total chunks in exceptions: {len(exceptions_chunked)}")
for chunk in exceptions_chunked[:6]:
    print(chunk)
    print()

Total chunks in exceptions: 5
{'chunk_id': 'exceptions.py_0', 'source_file': 'exceptions.py', 'start_line': 0, 'end_line': 39, 'text': '"""\nrequests.exceptions\n~~~~~~~~~~~~~~~~~~~\n\nThis module contains the set of Requests\' exceptions.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import TYPE_CHECKING, Any\n\nfrom urllib3.exceptions import HTTPError as BaseHTTPError\n\nfrom .compat import JSONDecodeError as CompatJSONDecodeError\n\nif TYPE_CHECKING:\n    from .models import PreparedRequest, Request, Response\n\n\nclass RequestException(IOError):\n    """There was an ambiguous exception that occurred while handling your\n    request.\n    """\n\n    response: Response | None\n    request: Request | PreparedRequest | None\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        """Initialize RequestException with `request` and `response` objects."""\n        response: Response | None = kwargs.pop("response", None)\n        self.response = response\n        s